In [ ]:
import os
import pandas as pd

# Existing noise TSV
noise_tsv = "/run/media/fourier/Data1/Pras/Interspeech2025/RIRS_NOISES/data_augmentation_noises_labels.tsv"
speech_root = "/run/media/fourier/Data1/Pras/Interspeech2025/Datasets/MELD"
out_tsv = "/run/media/fourier/Data1/Pras/Interspeech2025/RIRS_NOISES/data_augmentation_noises_speechs_labels.tsv"

# Load existing noise TSV
df_noise = pd.read_csv(noise_tsv, sep="\t", header=None, names=["filepath", "label"])

# Collect all speech WAV files recursively
speech_paths = []
for root, _, files in os.walk(speech_root):
    for f in files:
        if f.lower().endswith(".wav"):
            speech_paths.append(os.path.join(root, f))

# Build speech dataframe
df_speech = pd.DataFrame({
    "filepath": speech_paths,
    "label": ["speech"] * len(speech_paths)
})

# Append
df_speech = df_speech.sample(n=842, random_state=32)
df_out = pd.concat([df_noise, df_speech], ignore_index=True)

# Save final TSV
df_out.to_csv(out_tsv, sep="\t", index=False, header=False)

print(f"Appended and saved to: {out_tsv}")
print(f"Noise count: {len(df_noise)}")
print(f"Speech count: {len(df_speech)}")
print(f"Total: {len(df_out)}")


In [1]:
import argparse, inspect, json, os, pickle, socket, subprocess, warnings, random, math, librosa, shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from sklearn.metrics import accuracy_score, auc, balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score, roc_curve
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from tensorboard.backend.event_processing import event_accumulator
from torch.amp import GradScaler
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from transformers import AutoConfig, AutoFeatureExtractor, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split

import warnings
warnings.simplefilter("ignore", UserWarning)

def filter_participants(df, participant_col, file_col, min_files=5, max_files=15, seed=42):
    """
    Normalize participant-level sample counts.
    - Remove participants with < min_files
    - Randomly downsample participants with > max_files
    - Keep only participants within the [min_files, max_files] band
    """
    # Compute counts
    counts = df.groupby(participant_col)[file_col].count()

    # Identify cohorts
    drop_ids = counts[counts < min_files].index
    downsample_ids = counts[counts > max_files].index

    # Filter low-volume participants
    df_filtered = df[~df[participant_col].isin(drop_ids)]

    # Downsample oversized participants
    output_frames = []
    for pid, group in df_filtered.groupby(participant_col):
        if pid in downsample_ids:
            group = group.sample(n=max_files, random_state=seed)
        output_frames.append(group)

    return pd.concat(output_frames, ignore_index=True)


def balance_participants(
    df,
    participant_col="participant",
    label_col="disease_status",
    ratio=1.0,
    seed=42,
    minority_label=None,
):
    """
    Balances the dataset by sampling participants, not individual rows.

    minority_label:
        None → auto-detect minority class based on participant counts
        0 or 1 → force a specific class to be treated as minority
    """

    # Participant → label mapping
    p_map = df.groupby(participant_col)[label_col].first()

    # Participant counts per class
    counts = p_map.value_counts()

    # Determine minority size
    if minority_label is None:
        minority_size = counts.min()
    else:
        # Forced minority: its size = actual participant count of that label
        minority_size = counts[minority_label]

    # Cap for majority classes
    majority_cap = int(minority_size * ratio)

    keep_pids = []

    for label in counts.index:
        pids = p_map[p_map == label]

        if label == minority_label:
            # Keep minority participants fully
            keep_pids.extend(pids.index)
        else:
            # Apply capping to majority groups
            if len(pids) > majority_cap:
                pids = pids.sample(majority_cap, random_state=seed)
            keep_pids.extend(pids.index)

    return df[df[participant_col].isin(keep_pids)].reset_index(drop=True)

def sample_rows_by_participant(df, target_rows=3042, participant_col="participant", label_col="disease_status", random_state=42):
    """
    Sample rows from a dataframe per participant while roughly balancing classes.

    Args:
        df (pd.DataFrame): DataFrame containing participants and labels.
        target_rows (int): Approximate total number of rows to sample.
        participant_col (str): Column name for participant IDs.
        label_col (str): Column name for labels.
        random_state (int): Seed for reproducibility.

    Returns:
        pd.DataFrame: Sampled dataframe.
    """
    # Group participants by their label
    participants = df.groupby(participant_col)[label_col].first().reset_index()

    # Shuffle participants per class
    classes = participants[label_col].unique()
    class_participants = {}
    for c in classes:
        class_participants[c] = participants[participants[label_col] == c].sample(frac=1, random_state=random_state)

    # Helper to select participants until target rows reached
    def select_participants(part_df, target_rows):
        selected = []
        total_rows = 0
        for p in part_df[participant_col]:
            part_rows = len(df[df[participant_col] == p])
            if total_rows + part_rows > target_rows:
                break
            selected.append(p)
            total_rows += part_rows
        return selected

    # Roughly split target rows evenly across classes
    rows_per_class = target_rows // len(classes)
    selected_participants = []
    for c in classes:
        selected_participants += select_participants(class_participants[c], rows_per_class)

    # Filter original dataframe
    df_sampled = df[df[participant_col].isin(selected_participants)]

    return df_sampled


def assign_split_column(
    df,
    participant_col="participant",
    label_col="disease_status",
    train_ratio=0.7,  # adjusted to leave space for val
    val_ratio=0.1,    # fraction of participants for validation
    seed=42
):
    """
    Stratified participant split into train/val/test.
    Guarantees:
        • No participant overlap
        • Class-wise train/val/test ratio per participant
        • Approximate row-level ratios
    """
    # Participant → label
    p_map = df.groupby(participant_col)[label_col].first()

    train_pids = []
    val_pids = []
    test_pids = []

    rng = np.random.default_rng(seed)

    for label in p_map.unique():
        pids = p_map[p_map == label].index
        n_total = len(pids)
        
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)
        
        # shuffle participants
        shuffled = rng.permutation(pids)
        train_sel = shuffled[:n_train]
        val_sel = shuffled[n_train:n_train+n_val]
        test_sel = shuffled[n_train+n_val:]

        train_pids.extend(train_sel)
        val_pids.extend(val_sel)
        test_pids.extend(test_sel)

    # assign split
    df = df.copy()
    df["split"] = "train"
    df.loc[df[participant_col].isin(val_pids), "split"] = "val"
    df.loc[df[participant_col].isin(test_pids), "split"] = "test"

    return df


def encode_participants(df, participant_col="participant"):
    """
    Encode participants as numeric speaker IDs.

    Args:
        df (pd.DataFrame): DataFrame containing a participant column.
        participant_col (str): Column name for participant IDs.
        new_col (str): Name of the new numeric column.

    Returns:
        pd.DataFrame: DataFrame with an additional numeric speaker_id column.
    """
    df = df.copy()
    participant_mapping = {p: i for i, p in enumerate(df[participant_col].unique())}
    df[participant_col] = df[participant_col].map(participant_mapping)
    return df

In [2]:
df_longi = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/coda/longitudinal_original.csv')
df_longi = df_longi.rename(columns={"tb_status": "disease_status"})
#df_longi = filter_participants(df_longi, "participant", "path_file", min_files=10, max_files=15, seed=42)
#df_longi = balance_participants(df_longi, participant_col="participant", label_col="disease_status", ratio=0.5, minority_label=1, seed=42)
df_longi = assign_split_column(df_longi)
df_longi["db"] = 0
df_longi = df_longi[['path_file', 'participant', 'sex', 'disease_status', 'split', 'db']]
#print(df_longi['disease_status'].value_counts())

# df_solic = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/coda/solicited_original.csv')
# df_solic = df_solic.rename(columns={"tb_status": "disease_status"})
# df_solic = filter_participants(df_solic, "participant", "path_file", min_files=10, max_files=15, seed=42)
# df_solic = balance_participants(df_solic, participant_col="participant", label_col="disease_status", ratio=0.5, minority_label=1, seed=42)
# df_solic = assign_split_column(df_solic)
# df_solic["db"] = 1
# df_solic = df_solic[['path_file', 'participant', 'sex', 'disease_status', 'split', 'db']]
#print(df_solic['disease_status'].value_counts())

df_tbscreen_longi = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/TBscreen_Dataset/metadata_longitudinal.csv')
#df_tbscreen_longi = filter_participants(df_tbscreen_longi, "participant", "path_file", min_files=10, max_files=15, seed=42)
#df_tbscreen_longi = balance_participants(df_tbscreen_longi, participant_col="participant", label_col="disease_status", ratio=0.5, minority_label=1, seed=42)
df_tbscreen_longi = assign_split_column(df_tbscreen_longi)
df_tbscreen_longi["db"] = 2
df_tbscreen_longi = df_tbscreen_longi[['path_file', 'participant', 'sex', 'disease_status', 'split', 'db']]
#print(df_tbscreen_longi['disease_status'].value_counts())

# df_tbscreen_solic = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/TBscreen_Dataset/metadata_longitudinal.csv')
# df_tbscreen_solic = filter_participants(df_tbscreen_solic, "participant", "path_file", min_files=10, max_files=15, seed=42)
# df_tbscreen_solic = balance_participants(df_tbscreen_solic, participant_col="participant", label_col="disease_status", ratio=0.5, minority_label=1, seed=42)
# df_tbscreen_solic = assign_split_column(df_tbscreen_solic)
# df_tbscreen_solic["db"] = 3
# df_tbscreen_solic = df_tbscreen_solic[['path_file', 'participant', 'sex', 'disease_status', 'split', 'db']]
#print(df_tbscreen_solic['disease_status'].value_counts())

df_concat = pd.concat([df_longi, df_tbscreen_longi], axis=0, ignore_index=True)
df_concat['disease_status'].value_counts()
print(df_concat[df_concat['split'] == 'train']['disease_status'].value_counts())
print(df_concat[df_concat['split'] == 'val']['disease_status'].value_counts())
print(df_concat[df_concat['split'] == 'test']['disease_status'].value_counts())

disease_status
1    281965
0    175763
Name: count, dtype: int64
disease_status
0    16383
1    14700
Name: count, dtype: int64
disease_status
1    117826
0     65471
Name: count, dtype: int64


In [3]:
df_train_ukcovid = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/ukcovid19/metadata.csv.train')
df_val_ukcovid = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/ukcovid19/metadata.csv.val')
df_test_ukcovid = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/ukcovid19/metadata.csv.test')

df_train_ukcovid = df_train_ukcovid.rename(columns={"file_path": "path_file", 'covid_test_result': 'disease_status', 'gender': 'sex', 'participant_identifier': 'participant'})
df_val_ukcovid = df_val_ukcovid.rename(columns={"file_path": "path_file", 'covid_test_result': 'disease_status', 'gender': 'sex', 'participant_identifier': 'participant'})
df_test_ukcovid = df_test_ukcovid.rename(columns={"file_path": "path_file", 'covid_test_result': 'disease_status', 'gender': 'sex', 'participant_identifier': 'participant'})

# df_train_ukcovid = filter_participants(df_train_ukcovid, "participant", "path_file", min_files=10, max_files=15, seed=42)
# df_val_ukcovid = filter_participants(df_val_ukcovid, "participant", "path_file", min_files=10, max_files=15, seed=42)
# df_test_ukcovid = filter_participants(df_test_ukcovid, "participant", "path_file", min_files=10, max_files=15, seed=42)

# df_train_ukcovid = sample_rows_by_participant(df_train_ukcovid, target_rows=2241, participant_col="participant", label_col="disease_status")
# df_val_ukcovid = sample_rows_by_participant(df_val_ukcovid, target_rows=338, participant_col="participant", label_col="disease_status")
# df_test_ukcovid = sample_rows_by_participant(df_test_ukcovid, target_rows=593, participant_col="participant", label_col="disease_status")

df_train_ukcovid['split'] = "train"
df_val_ukcovid['split'] = "val"
df_test_ukcovid['split'] = "test"

df_ukcovid = pd.concat([df_train_ukcovid, df_val_ukcovid, df_test_ukcovid], axis=0, ignore_index=True)
df_ukcovid["db"] = 4
df_ukcovid['disease_status'] = 0

df_concat = pd.concat([df_concat, df_ukcovid], axis=0, ignore_index=True)
df_concat = encode_participants(df_concat)
df_concat['sex']  = df_concat['sex'].replace({'Male': 0, 'Female': 1})

print(df_concat['disease_status'].value_counts())

disease_status
0    434896
1    414491
Name: count, dtype: int64


/tmp/ipykernel_899211/2390221811.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_concat['sex']  = df_concat['sex'].replace({'Male': 0, 'Female': 1})


In [4]:
for split_name in df_concat['split'].unique():
    split_df = df_concat[df_concat['split'] == split_name].copy()
    filename = f"/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/metadata_mae.csv.{split_name}"
    split_df.to_csv(filename, index=False)
    print(f"Saved {split_name} split: {len(split_df)} rows → {filename}")

Saved train split: 556480 rows → /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/metadata_mae.csv.train
Saved test split: 237014 rows → /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/metadata_mae.csv.test
Saved val split: 55893 rows → /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/metadata_mae.csv.val


# Try Dataloader

In [25]:
%cd /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough

import os, pickle, random, json
import pandas as pd
import numpy as np
import hashlib

import torch
import torchaudio
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchaudio import transforms as T
import torch.nn.functional as F
import torchvision.transforms as transforms

import librosa
from sklearn.utils import shuffle 

import utils, commons, audio_processing
from augmentation import DataAugmentator, ISD_additive_noise, LnL_convolutive_noise
from cough_datasets import CoughDatasets, CoughDatasetsCollate, CoughDatasetsProcessorCollate
from scipy.signal import resample

from IPython.display import display, Audio
import matplotlib.pyplot as plt

def seed_from_path(path):
    h = hashlib.md5(path.encode()).hexdigest()
    seed = int(h[:8], 16) 
    random.seed(seed)
    np.random.seed(seed)

with open("/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/configs/general.json", "r") as f:
    data = f.read()

config = json.loads(data)
hps = utils.HParams(**config)

df_train = pd.read_csv('/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/metadata.csv.train')
if hps.data.reorder_target:
    cols = hps.data.column_order
    df_train = df_train[cols]

train_dataset = CoughDatasets(df_train.values, hps.data, train=True)
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=train_dataset.sampling_rate,
    n_fft=hps.data.filter_length,
    hop_length=train_dataset.hop_length,
    n_mels=hps.data.n_mel_channels
)

/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough
2.780925751721952e-05


In [ ]:
sample_data = df_train.sample(n=1).values[0][0]
audio_original = utils.load_audio_sample(sample_data, train_dataset.sampling_rate, train_dataset.saming_length, 
                                train_dataset.desired_length, fade_samples_ratio=train_dataset.fade_samples_ratio, 
                                pad_types="synthesis").squeeze().numpy()
#mel_original = torchaudio.functional.amplitude_to_DB(mel_transform(audio_original), multiplier=10, amin=1e-10, db_multiplier=0).squeeze().numpy()

In [ ]:
plt.plot(audio_original)

In [ ]:
# Skema1

In [ ]:
def apply_fade_in(audio, sr, sec=0.08):
    fade_len = int(sr * sec)
    fade_len = min(fade_len, len(audio))
    fade = np.linspace(0.0, 1.0, fade_len)
    audio[:fade_len] *= fade
    return audio

def apply_fade_out(audio, sr, sec=0.08):
    fade_len = int(sr * sec)
    fade_len = min(fade_len, len(audio))
    fade = np.linspace(1.0, 0.0, fade_len)
    audio[-fade_len:] *= fade
    return audio

def speed_perturb(audio, factor_min=0.9, factor_max=1.1):
    factor = random.uniform(factor_min, factor_max)
    new_len = int(len(audio) / factor)
    out = resample(audio, new_len)
    return np.clip(out, -1.0, 1.0)

def gain_perturb(audio, db_range):
    low, high = db_range
    gain_db = random.uniform(low, high)
    gain = 10 ** (gain_db / 20)
    out = audio * gain
    return np.clip(out, -1.0, 1.0)

def generate_gap(sr, low=0.08, high=0.12, noise_gain=0.0001):
    gap_sec = random.uniform(low, high)
    gap_len = int(sr * gap_sec)
    noise = np.random.randn(gap_len) * noise_gain
    return np.clip(noise, -1.0, 1.0)

def build_tail_segment(audio, sr):
    tail_gap = generate_gap(sr, low=0.15, high=0.4)
    tail = audio.copy()

    tail = speed_perturb(tail)
    tail = gain_perturb(tail, (-5.0, -1.0))
    return np.concatenate([tail_gap, tail], axis=0)

def augment_and_merge(audio_original, path, sr, gain_db_set=[(-5.0, -1.0)]):
    seed_from_path(path)

    # fade-out original
    audio_faded = audio_original.copy()
    audio_faded = apply_fade_out(audio_faded, sr)

    # extract segment aligned to energy peak
    sec_i_start = max((audio_original ** 2).argmax() - 2400, 0)
    sec_segment = audio_original.copy()[sec_i_start:]

    # fade-in
    sec_segment = apply_fade_in(sec_segment, sr)

    # speed perturbation
    sec_segment = speed_perturb(sec_segment)

    # gain perturbation
    gain_db = random.choice(gain_db_set)
    sec_segment = gain_perturb(sec_segment, gain_db)

    # noise gap prepend
    gap_seg = generate_gap(sr)
    sec_segment = np.concatenate([gap_seg, sec_segment], axis=0)

    if random.random() < 0.40:
        tail_segment = build_tail_segment(sec_segment, sr)
        sec_segment = np.concatenate([sec_segment, tail_segment], axis=0)

    # merge
    merged = np.concatenate([audio_faded, sec_segment], axis=0)
    return merged


In [ ]:
merged = augment_and_merge(
    audio_original,
    path=sample_data,
    sr=train_dataset.sampling_rate
)

merged = augment_and_merge(
    merged,
    path=sample_data,
    sr=train_dataset.sampling_rate
)

display(Audio(merged, rate=train_dataset.sampling_rate))
plt.plot(merged)


In [ ]:
0.1, 0.085

In [ ]:
plt.plot(audio_original ** 2)

In [ ]:
#audio_augment = getattr(train_dataset.data_augmentator, "add_background_noise")(audio_original, train_dataset.sampling_rate)
audio_augment = train_dataset.data_augmentator(audio_original, train_dataset.sampling_rate).squeeze(0)
mel_augment = torchaudio.functional.amplitude_to_DB(mel_transform(audio_augment), multiplier=10, amin=1e-10, db_multiplier=0).squeeze().numpy()

display(Audio(audio_original.squeeze().cpu().numpy(), rate=train_dataset.sampling_rate))
display(Audio(audio_augment.squeeze().cpu().numpy(), rate=train_dataset.sampling_rate))

In [ ]:


ipd.Audio(audio.squeeze().cpu().numpy(), rate=train_dataset.sampling_rate)




plt.figure(figsize=(10, 4))
plt.imshow(mel_db.squeeze().numpy(), aspect='auto', origin='lower')
plt.colorbar(label='dB')
plt.title("Mel-Spectrogram")
plt.xlabel("Frames")
plt.ylabel("Mel bins")
plt.tight_layout()
plt.show()

In [ ]:

        audio = audio.squeeze(0)
        if train_dataset.augment_data and train_dataset.train:
            if random.uniform(0, 0.999) > 1 - 0.8:
                try:
                    audio = train_dataset.data_augmentator(audio.unsqueeze(0), train_dataset.sampling_rate).squeeze(0)
                except:
                    audio = audio

        original_feature_dtype = audio.dtype
        if train_dataset.augment_rawboost and train_dataset.train:        
            feature = LnL_convolutive_noise(audio.numpy(), 5, 5, 20, 8000, 100, 1000,
                                            10, 100, 0, 0, 5, 20, train_dataset.sampling_rate)
            feature = ISD_additive_noise(feature, 10, 2)
            if not isinstance(feature, torch.Tensor):
                feature = torch.tensor(feature)
            if feature.dtype != original_feature_dtype:
                feature = feature.to(original_feature_dtype)
            audio = feature

        if train_dataset.add_noise:
            audio = audio + torch.rand_like(audio)

        audio = audio.unsqueeze(0)
        return audio

In [ ]:
sample_audio = train_dataset.get_audio(sample_data)

In [ ]:
sample_audio.shape

In [ ]:
sample_data

In [ ]:
audio = utils.load_audio_sample(sample_data, train_dataset.sampling_rate, train_dataset.saming_length, 
                                    train_dataset.desired_length, fade_samples_ratio=train_dataset.fade_samples_ratio, 
                                    pad_types=train_dataset.pad_types) # repeat zero
    audio = audio.squeeze(0)
    if train_dataset.augment_data and train_dataset.train:
        if random.uniform(0, 0.999) > 1 - 0.8:
            try:
                audio = train_dataset.data_augmentator(audio.unsqueeze(0), train_dataset.sampling_rate).squeeze(0)
            except:
                audio = audio

    original_feature_dtype = audio.dtype
    if train_dataset.augment_rawboost and train_dataset.train:        
        feature = LnL_convolutive_noise(audio.numpy(), 5, 5, 20, 8000, 100, 1000,
                                        10, 100, 0, 0, 5, 20, train_dataset.sampling_rate)
        feature = ISD_additive_noise(feature, 10, 2)
        if not isinstance(feature, torch.Tensor):
            feature = torch.tensor(feature)
        if feature.dtype != original_feature_dtype:
            feature = feature.to(original_feature_dtype)
        audio = feature

    if train_dataset.add_noise:
        audio = audio + torch.rand_like(audio)

    audio = audio.unsqueeze(0)
    return audio

In [ ]:
# check augmented data, check 1/1 how augmentation effect

In [ ]:
# Vizualize

cough_volume = (
    df_solic
      .groupby("participant")["path_file"]
      .count()
      .sort_values(ascending=False)
)

plt.figure(figsize=(14,4))
cough_volume.plot(kind="bar")
plt.title("Cough Count Distribution per Participant")
plt.xlabel("Participant")
plt.ylabel("Cough Count")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 2. COVID Test Result vs File Count
# ------------------------------------------------------------
covid_counts = (
    df_solic.groupby("disease_status")["path_file"]
      .count()
      .sort_values(ascending=False)
)

plt.figure(figsize=(6,4))
covid_counts.plot(kind="bar")
plt.title("File Volume by COVID Test Result")
plt.xlabel("COVID Test Result")
plt.ylabel("File Count")
plt.tight_layout()
plt.show()
